### Baseline vs. fine-tuned models evaluation

The purpose of this last notebook is to evaluate the fine-tuned `flan-t5-base` model across 4 quantization levels: FP32, BF16, INT8 and INT4.

The evaluation principles used here are the same as for the initial baseline evaluation:
*   same evaluation metrics:
    *    main metrics: Exact Match & F1;
    *    secondary metrics: Precision, Recall;
* same test split, tokenizer, quantization settings;
* same evaluation function `evaluate_model()` (inside of src/evaluate.py).

In [1]:
# github repo pull & dependencies installation for the evaluation
!git clone https://github.com/Julianlls/DeepLearning.git
%cd DeepLearning
!pip install -q transformers datasets accelerate bitsandbytes

Cloning into 'DeepLearning'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 122 (delta 58), reused 94 (delta 30), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 395.91 KiB | 3.41 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/DeepLearning
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.9 MB/s eta 0:00:00


To follow on the above-set conditions, in terms of the set-up we will :
*    import the src directory to reuse load_processed() and evaluate_model();
*    load the test split (3000 questions) using load_processed();
*    set the target hardware, using a GPU for INT8 and INT4 evaluation;
*    load the google/flan-t5-base tokenizer to maintain identical text tokenization.


In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

from src.evaluate import evaluate_model
from src.preprocessing import load_processed
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

ds = load_processed()
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/27000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

device: cuda


We also need to define how the fine-tuned model loads for each precision level we evaluate:

*   FP32 - standard loading using 32 bits per weight.
    *    our reference baseline;
*   BF16 - 16 bits per weight
    *    reduces the memory usage by half compared to FP32;
*   INT8 and INT4: 8 or 4 bits per weight using the bitsandbytes library.
    *    INT8 and INT4 models automatically load directly onto the GPU;

  
`load_model()` here loads the model at the specified quantization and transfers it to the GPU. It raises an error for unsupported quantization names.


In [3]:
FT_MODEL_ID = "Smambu/flan-t5-base-adversarialqa-ft"

KWARGS = {
    "fp32": {},
    "bf16": {"torch_dtype": torch.bfloat16},
    "int8": {"quantization_config": BitsAndBytesConfig(load_in_8bit = True), "device_map": "auto"},
    "int4": {"quantization_config": BitsAndBytesConfig(load_in_4bit = True), "device_map": "auto"},
}

def load_model(model_id, precision_mode):
    if precision_mode not in KWARGS:
        raise ValueError(f"Unknown precision_mode: {precision_mode!r}")

    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_id, **KWARGS[precision_mode]
    )

    if precision_mode in ("fp32", "bf16"):
        model = model.to(device)

    return model

We will skip the repeated bitsandbytes warnings during INT8/INT4 batch generation to keep the output clean without affecting the evaluation process; only genuine errors remain visible.

In [4]:
import logging
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

With the set-up being completed, we will now evaluate the fine-tuned model on the test set across all quantization levels.

We re-run FP32 here, even if it was previously evaluated in `04_finetuning` with F1 = 0.550 to ensure all fine-tuned results come from the same run and code path.

In [5]:
import pandas as pd

precisions = ["fp32", "bf16", "int8", "int4"]
rows = []

for precision_mode in precisions:
    print(f"--- flan-t5-base (fine-tuned) / {precision_mode} ---")

    model = load_model(FT_MODEL_ID, precision_mode)
    row, preds = evaluate_model(
        model, tokenizer, ds["test"],
        model_name="flan-t5-base",
        state="finetuned",
        precision_mode=precision_mode,
    )
    rows.append(row)
    print("\n", row, sep="")

    del model
    if device == 'mps': torch.mps.empty_cache()
    else: torch.cuda.empty_cache()

pd.DataFrame(rows)

--- flan-t5-base (fine-tuned) / fp32 ---


config.json:   0%|          | 0.00/1.56k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generating:   0%|          | 0/375 [00:00<?, ?it/s]


{'model_name': 'flan-t5-base', 'state': 'finetuned', 'precision_mode': 'int4', 'exact_match': 0.434, 'precision': 0.56058, 'recall': 0.5618, 'f1': 0.53785, 'n_examples': 3000}


,model_name,state,precision_mode,exact_match,precision,recall,f1,n_examples
0,flan-t5-base,finetuned,fp32,0.44400,0.56737,0.58402,0.55009,3000
1,flan-t5-base,finetuned,bf16,0.44233,0.56610,0.58378,0.54905,3000
2,flan-t5-base,finetuned,int8,0.44533,0.56744,0.58562,0.55128,3000
3,flan-t5-base,finetuned,int4,0.43400,0.56058,0.56180,0.53785,3000


The FP32 evaluation passed successfully. An Exact Match (EM) score of 0.444 matches the result from `04_finetuning`, confirming that the Hub checkpoint is identical to the model evaluated earlier.

We will now compare the baseline models and fine-tuned ones across all of our evaluation metrics.

In [8]:
# raw values taken from 03_baseline_evaluation for the baseline model
baseline = pd.DataFrame([
    {"precision_mode": "fp32", "exact_match": 0.42367, "precision": 0.55871, "recall": 0.55531, "f1": 0.53332},
    {"precision_mode": "bf16", "exact_match": 0.42567, "precision": 0.55996, "recall": 0.55622, "f1": 0.53420},
    {"precision_mode": "int8", "exact_match": 0.42200, "precision": 0.55709, "recall": 0.55543, "f1": 0.53232},
    {"precision_mode": "int4", "exact_match": 0.40333, "precision": 0.54037, "recall": 0.52487, "f1": 0.50944},
])

# values from the above-executed evaluation
finetuned = pd.DataFrame(rows)[["precision_mode", "exact_match", "f1"]]

comparison = baseline[["precision_mode", "exact_match", "f1"]].merge(
    finetuned, on="precision_mode", suffixes = ("_base", "_ft")
)

# calculation of the drop for each of the models vs. FP32
comparison["F1 vs. FP32 (base)"] = comparison["f1_base"] - comparison.loc[0, "f1_base"]
comparison["F1 vs. FP32 (ft)"] = comparison["f1_ft"] - comparison.loc[0, "f1_ft"]

# calculation of the gain from fine-tuning across each precision
comparison["F1 gain from fine-tuning"] = comparison["f1_ft"] - comparison["f1_base"]

comparison["precision_mode"] = comparison["precision_mode"].str.upper()
comparison = comparison.rename(columns={
    "precision_mode": "Quantization",
    "exact_match_base": "EM (base)",
    "exact_match_ft": "EM (ft)",
    "f1_base": "F1 (base)",
    "f1_ft": "F1 (ft)",
})

comparison.round(4)

,Quantization,EM (base),F1 (base),EM (ft),F1 (ft),F1 vs. FP32 (base),F1 vs. FP32 (ft),F1 gain from fine-tuning
0,FP32,0.4237,0.5333,0.4440,0.5501,0.0000,0.0000,0.0168
1,BF16,0.4257,0.5342,0.4423,0.5490,0.0009,-0.0010,0.0149
2,INT8,0.4220,0.5323,0.4453,0.5513,-0.0010,0.0012,0.0190
3,INT4,0.4033,0.5094,0.4340,0.5379,-0.0239,-0.0122,0.0284


*    base = baseline model (`google/flan-t5-base`)
*    ft = finetuned model (`Smambu/flan-t5-base-adversarialqa-ft`)

<br>

**Key evaluation comparison observations:**

*   Fine-tuning improves accuracy across all precision levels. At full precision (FP32), the fine-tuned model answers approximately 60 additional questions correctly out of 3000, increasing Exact Match accuracy from 0.424 to 0.444.
    *   Training on this dataset yields minor improvements, but overall gains are limited because the model saw the data during model's instruction tuning done by Google. We discovered this at the end of the project, so we could not redo the entire project with a different dataset.

*   BF16 and INT8 score changes remain minimal (around 0.001), representing roughly 3 questions out of 3000, which constitutes negligible noise.
    *    Model memory usage can be reduced by 2x (BF16) or 4x (INT8) with no real loss in accuracy, both before and after fine-tuning.

*   INT4 is the only precision level that causes performance to drop, despite significantly reducing the model's size compared to FP32, accuracy decreases for both models.
    *    There is a limit to how much you can compress the model without impacting performance, and INT4 reaches that limit.


*   The fine-tuned model drops half as much performance at INT4. Quantizing to INT4 reduces the raw model's accuracy by about 60 fewer correct answers overall (EM dropped from 0.424 to 0.403), whereas the fine-tuned model loses only about 30 (EM dropped from 0.444 to 0.434).
    *    Besides improving accuracy, fine-tuning made the model more resilient to heavy compression, making it a considerable choice in case strong quantization is needed.

*   The fine-tuned model answers about 90 more questions correctly than the raw model at INT4, compared to a gain of 60 questions at FP32.
    *    Fine-tuning delivered its best improvement where compression was the highest.

*   A fine-tuned INT4 model does at least as well as the base FP32 model (F1 0.538 vs. 0.533 & EM 0.434 vs. 0.424). The fine-tuned INT4 model achieves an F1 score of 0.538, outperforming the base FP32 model at 0.533.
    *    The gain from fine-tuning (F1 +0.017) is larger than the loss from INT4 compression (F1 -0.012), so aggresive quantization applied (4 bits per weight instead of 32) does not lead to drastical loss the accuracy compared to the original full-sized model.